In [1]:
import pandas as pd           # manipulación de DataFrames y lectura de archivos
import numpy as np             # operaciones numéricas sobre arrays
import requests                # realiza peticiones HTTP a APIs externas (GET, POST, etc.)
import json                    # serializa y deserializa datos en formato JSON
import io                      # maneja flujos de bytes en memoria (útil para simular archivos)
import time                    # mide tiempos de ejecución de cada fase del pipeline
from pathlib import Path       # manejo de rutas de archivos de forma multiplataforma
import glob
import os

print("✅ Librerías cargadas")

c:\Users\JORGE\miniconda3\envs\py_env\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


✅ Librerías cargadas


In [2]:
# Creamos un CSV sucio en memoria usando io.StringIO
# En producción esto vendría de: pd.read_csv('ruta/al/archivo.csv')

raw_csv = """id,empleado,departamento,pais,salario,fecha_ingreso,edad,activo,email
1,Ana García,Ventas,Colombia,$ 4.500,2021-03-15,28,True,ana.garcia@empresa.com
2,CARLOS LOPEZ,ventas,colombia,3200.50,15/04/2020,35,True,carlos.lopez@empresa.com
3,Ana García,Ventas,Colombia,$ 4.500,2021-03-15,28,True,ana.garcia@empresa.com
4,María Rodríguez,RRHH,México,$ 5.800,2019-07-01,42,False,maria.rodriguez@empresa.com
5,Pedro Martínez,,Colombia,2100,-5,True,pedro.martinez@empresa.com
6,Lucía Fernández,TI,Argentina,,2023-01-10,31,True,lucia.fernandez@empresa.com
7,PEDRO MARTINEZ,Operaciones,colombia,04/22/2022,$ 2.100,38,True,pedro.martinez@empresa.com
8,Jorge Ramírez,TI,Colombia,$ 6.700,,0,True,jorge.ramirez@empresa.com
9,Sofia Castro,Ventas,México,$ 3.900,2022-11-30,26,True,sofia.castro@empresa.com
10,Andrés Molina,rrhh,Argentina,$ 4.100,2020-05-18,None,False,andres.molina@empresa.com
11,Valentina Cruz,TI,Colombia,$ 7.200,2018-09-25,39,True,valentina.cruz@empresa.com
12,SOFIA CASTRO,Ventas,México,$ 3.900,2022-11-30,26,True,sofia.castro@empresa.com
"""

def extract_csv_chunks(source, chunksize=4):
    
    chunks = []
    n_total = 0
    
    lector = pd.read_csv(source, chunksize=chunksize)
    
    for num, chunk in enumerate(lector, start=1):
        n_total += len(chunk)
        chunks.append(chunk) #Acumlamos el bloque
        print(f" Chunk {num}: {len(chunk)} filas | acumulado: {n_total} filas")
    
    df = pd.concat(chunks, ignore_index=True)
    return df

print("=== Extrayendo CSV en chunks de 4 filas ===")
buffer = io.StringIO(raw_csv)
df_csv = extract_csv_chunks(buffer, chunksize=4)

print(f" Total extraido: {df_csv.shape[0]} filas x {df_csv.shape[1]} columnas")
df_csv

=== Extrayendo CSV en chunks de 4 filas ===
 Chunk 1: 4 filas | acumulado: 4 filas
 Chunk 2: 4 filas | acumulado: 8 filas
 Chunk 3: 4 filas | acumulado: 12 filas
 Total extraido: 12 filas x 9 columnas


,id,empleado,departamento,pais,salario,fecha_ingreso,edad,activo,email
0,1,Ana García,Ventas,Colombia,$ 4.500,2021-03-15,28,True,ana.garcia@empresa.com
1,2,CARLOS LOPEZ,ventas,colombia,3200.50,15/04/2020,35,True,carlos.lopez@empresa.com
2,3,Ana García,Ventas,Colombia,$ 4.500,2021-03-15,28,True,ana.garcia@empresa.com
3,4,María Rodríguez,RRHH,México,$ 5.800,2019-07-01,42,False,maria.rodriguez@empresa.com
4,5,Pedro Martínez,NaN,Colombia,2100,-5,True,pedro.martinez@empresa.com,NaN
5,6,Lucía Fernández,TI,Argentina,NaN,2023-01-10,31,True,lucia.fernandez@empresa.com
6,7,PEDRO MARTINEZ,Operaciones,colombia,04/22/2022,$ 2.100,38,True,pedro.martinez@empresa.com
7,8,Jorge Ramírez,TI,Colombia,$ 6.700,NaN,0,True,jorge.ramirez@empresa.com
8,9,Sofia Castro,Ventas,México,$ 3.900,2022-11-30,26.0,True,sofia.castro@empresa.com
9,10,Andrés Molina,rrhh,Argentina,$ 4.100,2020-05-18,NaN,False,andres.molina@empresa.com


## Extraer multiples archivos en un mismo momento

In [3]:
def extract_multiples_achivos(patron_glob):
    
    archivos = glob.glob(patron_glob)
    
    if not archivos:
        print(f" No se encontraron arhcivos con ese patrón: {patron_glob}")
        return pd.DataFrame()
    
    print(f" Archivos encontrados: {len(archivos)}")
    dfs = []
    
    for ruta in sorted(archivos):
        df_tmp = pd.read_csv(ruta)
        
        df_tmp['archivo_origen'] = Path(ruta).name
        
        dfs.append(df_tmp)
        print(f" {Path(ruta).name}: {len(df_tmp)} filas")
    
    df_combinado = pd.concat(dfs, ignore_index=True)
    return df_combinado

#Simulación
os.makedirs('data_lab', exist_ok=True)

datos_mes = {
    "data_lab/empleados_2024_01.csv": """id,nombre,salario
1,Alice,3000
2,Bob,3500
3,Carol,4000
4,David,2800""",

    "data_lab/empleados_2024_02.csv": """id,nombre,salario
3,Carol,4000
4,David,2800
5,Eve,5000
6,Frank,3200""",

    "data_lab/empleados_2024_03.csv": """id,nombre,salario
5,Eve,5000
6,Frank,3200"""
}

#Escribir lo que es los archivos temporales en el disco
for ruta, contenido in datos_mes.items():
    with open(ruta, 'w') as f:
        f.write(contenido)

print("=== Leyendo multiples archivos ===")
df_multi = extract_multiples_achivos("data_lab/*.csv")

print(f"DataFrame combinado {df_multi.shape}")
df_multi

=== Leyendo multiples archivos ===
 Archivos encontrados: 3
 empleados_2024_01.csv: 4 filas
 empleados_2024_02.csv: 4 filas
 empleados_2024_03.csv: 2 filas
DataFrame combinado (10, 4)


,id,nombre,salario,archivo_origen
0,1,Alice,3000,empleados_2024_01.csv
1,2,Bob,3500,empleados_2024_01.csv
2,3,Carol,4000,empleados_2024_01.csv
3,4,David,2800,empleados_2024_01.csv
4,3,Carol,4000,empleados_2024_02.csv
5,4,David,2800,empleados_2024_02.csv
6,5,Eve,5000,empleados_2024_02.csv
7,6,Frank,3200,empleados_2024_02.csv
8,5,Eve,5000,empleados_2024_03.csv
9,6,Frank,3200,empleados_2024_03.csv


In [4]:
def extract_api(base_url, params_base= None, max_paginas=3, pause_entre_pagina=0.5):
    
    params_base = params_base or {}
    todos_registros = []
    
    for pagina in range(1, max_paginas + 1):
        params = {
            **params_base,
            '_start': (pagina - 1) * 10,
            '_limit': 10
        }
        
        try:
            respuesta = requests.get(base_url, params=params, timeout=10)
            
            respuesta.raise_for_status()
            
            datos = respuesta.json()
            
            if not datos:
                print(f" Pagina {pagina}: sin datos - fin de resultados")
                break
            
            todos_registros.extend(datos)
            print(f" Pagina {pagina}: {len(datos)} registros | acumulado: {len(todos_registros)}")
            
            
            if pagina < max_paginas:
                time.sleep(pause_entre_pagina)
        
        except requests.exceptions.ConnectionError:
            print(f" Sin conexión wifi. Saltando extracción.....")
            break
        
        except requests.exceptions.Timeout:
            print(f" Timeout en pagina {pagina}. El servidor no respondió.")
            break
        
        except requests.exceptions.HTTPError as e:
            print(f" Error HTTP {respuesta.status_code}: {e}")
            break
    
    if not todos_registros:
        return pd.DataFrame()
    
    return pd.json_normalize(todos_registros)


print(" === Extrayendo datos de la API ===")

df_api = extract_api(
    base_url='https://jsonplaceholder.typicode.com/users',
    max_paginas=2,
    pause_entre_pagina=0.3
)

if not df_api.empty:
    print(f" API: {df_api.shape[0]} registros x {df_api.shape[1]} columnas")
    print("Columnas:", df_api.columns.tolist())
    display(df_api[['id', 'name', 'email', 'phone', 'company.name']].head(8))

else:
    print("La API no tiene datos...............")

 === Extrayendo datos de la API ===
 Pagina 1: 10 registros | acumulado: 10
 Pagina 2: sin datos - fin de resultados
 API: 10 registros x 15 columnas
Columnas: ['id', 'name', 'username', 'email', 'phone', 'website', 'address.street', 'address.suite', 'address.city', 'address.zipcode', 'address.geo.lat', 'address.geo.lng', 'company.name', 'company.catchPhrase', 'company.bs']


,id,name,email,phone,company.name
0,1,Leanne Graham,Sincere@april.biz,1-770-736-8031 x56442,Romaguera-Crona
1,2,Ervin Howell,Shanna@melissa.tv,010-692-6593 x09125,Deckow-Crist
2,3,Clementine Bauch,Nathan@yesenia.net,1-463-123-4447,Romaguera-Jacobson
3,4,Patricia Lebsack,Julianne.OConner@kory.org,493-170-9623 x156,Robel-Corkery
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,(254)954-1289,Keebler LLC
5,6,Mrs. Dennis Schulist,Karley_Dach@jasper.info,1-477-935-8478 x6430,Considine-Lockman
6,7,Kurtis Weissnat,Telly.Hoeger@billy.biz,210.067.6132,Johns Group
7,8,Nicholas Runolfsdottir V,Sherwood@rosamond.me,586.493.6943 x140,Abernathy Group


In [5]:
def limpiar(df):
    
    df = df.copy()   # siempre trabajamos sobre una copia — el original es inmutable
    print("--- Limpieza ---")

    strings_nulo = ['None', 'none', 'null', 'NULL', 'N/A', 'n/a', '-', '']
    df = df.replace(strings_nulo, np.nan)

    n_antes = len(df)
    df = df.drop_duplicates(keep='first').reset_index(drop=True)
    print(f"  Duplicados eliminados: {n_antes - len(df)}  ({n_antes} → {len(df)} filas)")

    n_antes = len(df)
    df = df.dropna(subset=['empleado', 'departamento'])
    print(f"  Filas sin campos críticos eliminadas: {n_antes - len(df)}")

    df['edad'] = pd.to_numeric(df['edad'], errors='coerce')

    mediana_edad = df.loc[df['edad'].between(18, 70), 'edad'].median()
    df['edad'] = df['edad'].fillna(mediana_edad)
    print(f"  Nulos en edad imputados con mediana: {mediana_edad:.0f}")

    df['salario'] = (
        df['salario'].astype(str)
        .str.replace('$', '', regex=False)    # elimina el símbolo de dólar/peso
        .str.replace(' ', '', regex=False)    # elimina espacios en blanco
        .str.replace('.', '', regex=False)    # elimina puntos de miles
        .str.strip()                          # elimina espacios al inicio y fin
    )
    df['salario'] = pd.to_numeric(df['salario'], errors='coerce')

    # Imputar nulos de salario con la mediana de salarios positivos
    mediana_sal = df.loc[df['salario'] > 0, 'salario'].median()
    df['salario'] = df['salario'].fillna(mediana_sal)
    print(f"  Nulos en salario imputados con mediana: {mediana_sal:,.0f}")

    print(f"  Resultado limpieza: {len(df)} filas")
    return df


df_limpio = limpiar(df_csv)
print()
df_limpio

--- Limpieza ---
  Duplicados eliminados: 0  (12 → 12 filas)
  Filas sin campos críticos eliminadas: 1
  Nulos en edad imputados con mediana: 31
  Nulos en salario imputados con mediana: 4,500
  Resultado limpieza: 11 filas



,id,empleado,departamento,pais,salario,fecha_ingreso,edad,activo,email
0,1,Ana García,Ventas,Colombia,4500.0,2021-03-15,28.0,True,ana.garcia@empresa.com
1,2,CARLOS LOPEZ,ventas,colombia,320050.0,15/04/2020,35.0,True,carlos.lopez@empresa.com
2,3,Ana García,Ventas,Colombia,4500.0,2021-03-15,28.0,True,ana.garcia@empresa.com
3,4,María Rodríguez,RRHH,México,5800.0,2019-07-01,42.0,False,maria.rodriguez@empresa.com
5,6,Lucía Fernández,TI,Argentina,4500.0,2023-01-10,31.0,True,lucia.fernandez@empresa.com
6,7,PEDRO MARTINEZ,Operaciones,colombia,4500.0,$ 2.100,38.0,True,pedro.martinez@empresa.com
7,8,Jorge Ramírez,TI,Colombia,6700.0,NaN,0.0,True,jorge.ramirez@empresa.com
8,9,Sofia Castro,Ventas,México,3900.0,2022-11-30,26.0,True,sofia.castro@empresa.com
9,10,Andrés Molina,rrhh,Argentina,4100.0,2020-05-18,31.0,False,andres.molina@empresa.com
10,11,Valentina Cruz,TI,Colombia,7200.0,2018-09-25,39.0,True,valentina.cruz@empresa.com


In [7]:
def normalizar(df):
    
    df = df.copy()
    print("--- Normalización ---")

    
    for col in ['empleado', 'departamento', 'pais']:
        if col in df.columns:
            df[col] = df[col].str.strip().str.title()

    print(f"  Departamentos: {sorted(df['departamento'].unique())}")
    print(f"  Países:        {sorted(df['pais'].unique())}")

  
    df['fecha_ingreso'] = pd.to_datetime(
        df['fecha_ingreso'],
        format='mixed',
        dayfirst=True,
        errors='coerce'
    )

    # Cuántas fechas no se pudieron parsear (NaT)
    n_nat = df['fecha_ingreso'].isna().sum()
    print(f"  Fechas no parseables (NaT): {n_nat}")

    # Extraemos columnas derivadas útiles para análisis
    df['anio_ingreso']   = df['fecha_ingreso'].dt.year          # año como entero
    df['antiguedad_dias'] = (pd.Timestamp.now() - df['fecha_ingreso']).dt.days  # días desde ingreso

   
    mapa_bool = {True: True, False: False, 'True': True, 'False': False,
                 '1': True, '0': False, 1: True, 0: False}
    df['activo'] = df['activo'].map(mapa_bool)

   
    if 'email' in df.columns:
        df['email'] = df['email'].str.lower().str.strip()

    print(f"  Normalización completada: {len(df)} filas")
    return df


df_norm = normalizar(df_limpio)
print()
df_norm[['empleado','departamento','pais','fecha_ingreso','anio_ingreso',
         'antiguedad_dias','salario','edad','activo']].head(10)

--- Normalización ---
  Departamentos: ['Operaciones', 'Rrhh', 'Ti', 'Ventas']
  Países:        ['Argentina', 'Colombia', 'México']
  Fechas no parseables (NaT): 2
  Normalización completada: 11 filas



,empleado,departamento,pais,fecha_ingreso,anio_ingreso,antiguedad_dias,salario,edad,activo
0,Ana García,Ventas,Colombia,2021-03-15,2021.0,2005.0,4500.0,28.0,True
1,Carlos Lopez,Ventas,Colombia,2020-04-15,2020.0,2339.0,320050.0,35.0,True
2,Ana García,Ventas,Colombia,2021-03-15,2021.0,2005.0,4500.0,28.0,True
3,María Rodríguez,Rrhh,México,2019-07-01,2019.0,2628.0,5800.0,42.0,False
5,Lucía Fernández,Ti,Argentina,2023-01-10,2023.0,1339.0,4500.0,31.0,True
6,Pedro Martinez,Operaciones,Colombia,NaT,NaN,NaN,4500.0,38.0,True
7,Jorge Ramírez,Ti,Colombia,NaT,NaN,NaN,6700.0,0.0,True
8,Sofia Castro,Ventas,México,2022-11-30,2022.0,1380.0,3900.0,26.0,True
9,Andrés Molina,Rrhh,Argentina,2020-05-18,2020.0,2306.0,4100.0,31.0,False
10,Valentina Cruz,Ti,Colombia,2018-09-25,2018.0,2907.0,7200.0,39.0,True


# Validación de Esquemas

In [8]:
def validar_esquema(df):
    
    errores = []
    warnings_ = []
    
    cols_req = ['id', 'empleado', 'departamento', 'pais', 'salario', 'edad', 'activo']
    faltantes = [c for c in cols_req if c not in df.columns]
    if faltantes:
        raise ValueError(f" Columnnas requeridad faltantes: {faltantes}")
    
    
    #Sin nulos en campos críticos
    for col in ['id', 'empleado', 'departamento']:
        n = df[col].isna().sum()
        if n > 0:
            errores.append(f" '{col}' tiene {n} valores nulos después de la limpieza")
    
    # id debe ser único sin duplicados
    n_dup = df['id'].duplicated().sum()
    if n_dup > 0:
        errores.append(f" 'id' tiene {n_dup} valores duplicados - violar la clave primaria")
        
    
    # Salario en rango razonable:
    n_sal_negativo = (df['salario'] <= 0).sum()
    n_sal_alto     = (df['salario'] > 50_000).sum()
    
    if n_sal_negativo > 0:
        errores.append(f" 'Salario' tiene {n_sal_negativo} valores <= 0")
    
    if n_sal_alto > 0:
        warnings_.append(f" 'Salario' tiene {n_sal_alto} valores > 50.000 (revisión manual)")
    
    # Edad en rango laboral
    n_edad = (df['edad'].between(18, 75)).sum()
    if n_edad > 0:
        errores.append(f" 'Edad' tiene {n_edad} valores fuera dle rango laboral [18,75]")
        
    
    #Departamanetos en lista  de valores permitidos
    depts_validos = {'Ventas', 'Rrhh', 'Ti', 'Operaciones', 'Operaciones', 'Finanzas', 'Marketing'}
    depts_en_datos = set(df['departamento'].dropna().unique())
    depts_invalidos = depts_en_datos - depts_validos
    if depts_invalidos:
        warnings_.append(f"Departamentos no reconocidos: {depts_invalidos}")
        
    
    #Resultado
    if warnings_:
        print(" Advertencias (no críticas)")
        for w in warnings_:
            print(f"  {w}")
    
    print("Esquema válido - todos los check pasaron:")
    checks = ['Columnas requeridas presentes', 'Sin nulos en campos críticos', 'IDs únicos',
              'salarios positivos', 'Edades en rango [18-75]']
    
    for c in checks:
        print(f" {c}")
    return True


df_norm['salario'] = df_norm['salario'].apply(lambda s: df_norm.loc[df_norm['salario'] > 0, 'salario'].median()
                                              if pd.notna(s) and s <= 0 else s)

print("Validando Esquema.....")
try:
    validar_esquema(df_norm)
except ValueError as e:
    print(e)

Validando Esquema.....
 Advertencias (no críticas)
   'Salario' tiene 1 valores > 50.000 (revisión manual)
Esquema válido - todos los check pasaron:
 Columnas requeridas presentes
 Sin nulos en campos críticos
 IDs únicos
 salarios positivos
 Edades en rango [18-75]


### Load: Guardar el Resultado

Si no hay Load, el ETL no entrega nada: `df_norm` vive solo en memoria y se pierde al cerrar el kernel.

Load no transforma. Toma la tabla ya limpia y validada y la persiste. El crudo se queda en `data_lab/`; el resultado limpio va a `output/`.

El único input de esta fase es `df_norm`. No se vuelven a guardar `df_csv`, `df_limpio`, `df_multi` ni `df_api`.

Misma tabla, dos formatos:

- **CSV** — texto, se abre en Excel. Destino de intercambio. Al releer, `fecha_ingreso` vuelve a texto (`object`).
- **Parquet** — binario columnar. Destino analítico: comprime, conserva `datetime` / `float` / `bool`. Requiere `pyarrow`.

Parquet no es otra tabla: es otra forma de guardar la misma `df_norm`.

In [9]:
def load_csv(df, ruta):
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(ruta, index=False, encoding='utf-8')
    peso_kb = ruta.stat().st_size / 1024
    print(f" CSV     → {ruta} | {len(df)} filas | {peso_kb:.2f} KB")
    return ruta


def load_parquet(df, ruta):
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)

    try:
        df.to_parquet(ruta, index=False)
        peso_kb = ruta.stat().st_size / 1024
        print(f" Parquet → {ruta} | {len(df)} filas | {peso_kb:.2f} KB")
        return ruta
    except ImportError:
        print(" pyarrow no está instalado. Ejecuta: pip install pyarrow")
        return None


print("=== Load: persistiendo df_norm ===")
os.makedirs('output', exist_ok=True)

ruta_csv     = load_csv(df_norm, 'output/empleados_limpios.csv')
ruta_parquet = load_parquet(df_norm, 'output/empleados_limpios.parquet')

=== Load: persistiendo df_norm ===
 CSV     → output\empleados_limpios.csv | 11 filas | 1.14 KB
 Parquet → output\empleados_limpios.parquet | 11 filas | 7.24 KB


Load no termina al escribir. Hay que releer y comprobar que las filas se conservan. El formato cambia los tipos, no la tabla: `fecha_ingreso` en CSV vuelve a texto; en Parquet sigue siendo fecha.

In [10]:
print("--- Verificación round-trip ---")
print(f" Origen  : {df_norm.shape} | fecha_ingreso = {df_norm['fecha_ingreso'].dtype}")

df_desde_csv = pd.read_csv(ruta_csv)
print(f" CSV     : {df_desde_csv.shape} | fecha_ingreso = {df_desde_csv['fecha_ingreso'].dtype}")

if ruta_parquet and Path(ruta_parquet).exists():
    df_desde_parquet = pd.read_parquet(ruta_parquet)
    print(f" Parquet : {df_desde_parquet.shape} | fecha_ingreso = {df_desde_parquet['fecha_ingreso'].dtype}")
else:
    print(" Parquet : no disponible (se omitió la verificación)")

--- Verificación round-trip ---
 Origen  : (11, 11) | fecha_ingreso = datetime64[ns]
 CSV     : (11, 11) | fecha_ingreso = object
 Parquet : (11, 11) | fecha_ingreso = datetime64[ns]
